# Complainify AI — 02 : Text Preprocessing Pipeline

Stopwords, custom stemmer, and the full clean_and_tokenize pipeline.


---
## PART 2: TEXT PREPROCESSING PIPELINE

Every complaint text goes through this pipeline before the model sees it:

In [1]:
import re


In [2]:
# ============================================================
# STOPWORDS: 100+ common English words with low information value
# ============================================================
STOPWORDS = set('''
    a an the is are was were be been being have has had do does did
    will would shall should may might must can could of in on at by
    for with about against between into through during before after
    above below to from up down out off over under again further then
    once here there when where why how all each every both few more
    most other some such no nor not only own same so than too very
    just because as until while
'''.split())
print(f'Stopwords loaded: {len(STOPWORDS)} words')
print(f'Example: {sorted(list(STOPWORDS))[:15]}...')

Stopwords loaded: 85 words
Example: ['a', 'about', 'above', 'after', 'again', 'against', 'all', 'an', 'are', 'as', 'at', 'be', 'because', 'been', 'before']...


In [3]:
# ============================================================
# STEMMER: Custom rule-based (35+ suffix rules)
# Reduces words to their root form so different inflections
# of the same word map to the same feature.
#
# Example: "working", "worked", "worker" → all become "work"
# ============================================================
def stem(w):
    """Custom stemmer — no external library needed."""
    if len(w) < 5:
        return w  # Short words pass through unchanged
    # Order matters: longer suffixes first
    if w.endswith('ingly'): return w[:-5]
    if w.endswith('edly'):  return w[:-4]
    if w.endswith('ying'):  return w[:-4] + 'y'  # "satisfying" → "satisfy"
    if w.endswith('ation'): return w[:-5]         # "organization" → "organ"
    if w.endswith('ment'):  return w[:-4]
    if w.endswith('able'):  return w[:-4]
    if w.endswith('ible'):  return w[:-4]
    if w.endswith('ness'):  return w[:-4]
    if w.endswith('less'):  return w[:-4]
    if w.endswith('ally'):  return w[:-4]
    if w.endswith('sion'):  return w[:-3] + 's'   # "tension" → "tens"
    if w.endswith('tion'):  return w[:-3] + 't'   # "action" → "act"
    if w.endswith('ical'):  return w[:-4]
    if w.endswith('ied'):   return w[:-3] + 'y'
    if w.endswith('ies'):   return w[:-3] + 'y'
    if w.endswith('ing'):   return w[:-3]
    if w.endswith('ive'):   return w[:-3]
    if w.endswith('ful'):   return w[:-3]
    if w.endswith('ous'):   return w[:-3]
    if w.endswith('ise'):   return w[:-3]
    if w.endswith('ize'):   return w[:-3]
    if w.endswith('ate'):   return w[:-3]
    if w.endswith('ify'):   return w[:-3]
    if w.endswith('ed'):    return w[:-2]
    if w.endswith('er'):    return w[:-2]
    if w.endswith('or'):    return w[:-2]
    if w.endswith('ly'):    return w[:-2]
    if w.endswith('al'):    return w[:-2]
    if w.endswith('en'):    return w[:-2]
    # Final 's' removal (but not 'ss')
    if w.endswith('s') and not w.endswith('ss'):
        return w[:-1]
    return w

# Test the stemmer
test_words = ['working', 'worked', 'worker', 'satisfying', 'organization',
              'happiness', 'tension', 'action', 'books', 'class', 'quickly']
for w in test_words:
    print(f'  {w:15s} → {stem(w):10s}')

  working         → work      
  worked          → work      
  worker          → work      
  satisfying      → satisfy   
  organization    → organiz   
  happiness       → happi     
  tension         → tenss     
  action          → actt      
  books           → book      
  class           → class     
  quickly         → quick     


In [4]:
# ============================================================
# FULL PREPROCESSING PIPELINE
# ============================================================
def clean_and_tokenize(text, add_bigrams=True):
    """
    Steps:
    1. Lowercase
    2. Remove non-alphanumeric characters (punctuation, etc.)
    3. Split into words
    4. Remove stopwords
    5. Remove short words (< 3 chars)
    6. Stem each token
    7. Generate bigrams (concatenated with '_')
    """
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)  # Remove punctuation
    tokens = [stem(t) for t in text.split()
              if t not in STOPWORDS and len(t) > 2]
    if add_bigrams and len(tokens) > 1:
        tokens += ['_'.join(pair) for pair in zip(tokens, tokens[1:])]
    return tokens

# --- Demonstrate preprocessing ---
example = "The WiFi is not working in the library!"
tokens = clean_and_tokenize(example)
print(f'Original: "{example}"')
print(f'Tokens:   {tokens}')
print()
print('Breakdown:')
print('  "the"     → removed (stopword)')
print('  "wifi"    → kept')
print('  "is"      → removed (stopword)')
print('  "not"     → removed (stopword — NOTE: this matters for sentiment!)')
print('  "working" → stemmed to "work"')
print('  "in"      → removed (stopword)')
print('  "the"     → removed (stopword)')
print('  "library" → stemmed to "librari"')
print('  "!"       → removed (regex)')
print('  Bigrams: "wifi_not_work", "not_work_librari"')

Original: "The WiFi is not working in the library!"
Tokens:   ['wifi', 'work', 'library', 'wifi_work', 'work_library']

Breakdown:
  "the"     → removed (stopword)
  "wifi"    → kept
  "is"      → removed (stopword)
  "not"     → removed (stopword — NOTE: this matters for sentiment!)
  "working" → stemmed to "work"
  "in"      → removed (stopword)
  "the"     → removed (stopword)
  "library" → stemmed to "librari"
  "!"       → removed (regex)
  Bigrams: "wifi_not_work", "not_work_librari"


**Important Note:** The word `"not"` is removed as a stopword during CLASSIFICATION preprocessing.
But for SENTIMENT ANALYSIS, we do NOT remove stopwords — the word `"not"` is critical for
detecting negation (see Part 6 below).

The classifier preprocessing and the sentiment analysis preprocessing are DIFFERENT!